# 数据超分辨率测试

In [2]:
import sys
import os

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 强制重新加载模块（避免缓存问题）
import importlib
import olmoearth_pretrain.data.constants
importlib.reload(olmoearth_pretrain.data.constants)


from pathlib import Path
from test_planet_rgbnir_real_data import super_resolution_test, TEST_DATA_DIR

# 获取测试文件
h5_files = sorted(TEST_DATA_DIR.glob("*.h5"))
test_file = h5_files[0]

# 执行超分辨率并写入新文件
success = super_resolution_test(test_file, dry_run=True)

if success:
    output_path = test_file.parent / "planet_rgbnir_output" / test_file.name
    print(f"✅ 成功! 输出文件: {output_path}")


超分辨率测试: sample_0.h5
模式: DRY RUN
输出文件: /mnt/ht2-nas2/00-model/00-jiangzf/coderepo/3996/planet_rgbnir_output/sample_0.h5

✅ 加载 sentinel2_l2a: shape=(128, 128, 12, 12), dtype=uint16
✅ RGB波段索引: R=B04(2), G=B03(1), B=B02(0)
✅ 提取RGB: shape=(128, 128, 12, 3)
✅ 缩放因子: 4.00 (256 -> 1024)

🔄 执行超分辨率: (128, 128) -> (512, 512)
✅ 超分辨率完成: shape=(512, 512, 12, 3), dtype=uint16
   数据范围: [0, 65535]

⏭️  Dry run模式,跳过写入
✅ 成功! 输出文件: /mnt/ht2-nas2/00-model/00-jiangzf/coderepo/3996/planet_rgbnir_output/sample_0.h5


In [ ]:
import sys
import os

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 强制重新加载模块（避免缓存问题）
import importlib
import olmoearth_pretrain.data.constants
importlib.reload(olmoearth_pretrain.data.constants)

from pathlib import Path
from test_planet_rgbnir_real_data import super_resolution_test, TEST_DATA_DIR

# 获取所有H5文件
h5_files = sorted(TEST_DATA_DIR.glob("*.h5"))
print(f"找到 {len(h5_files)} 个文件")

# 批量处理
success_count = 0
for h5_file in h5_files:
    print(f"\n处理: {h5_file.name}")
    if super_resolution_test(h5_file, dry_run=False):
        success_count += 1

print(f"\n完成! 成功: {success_count}/{len(h5_files)}")

找到 3996 个文件

处理: sample_0.h5

超分辨率测试: sample_0.h5
模式: WRITE
输出文件: /mnt/ht2-nas2/00-model/00-jiangzf/coderepo/3996/planet_rgbnir_output/sample_0.h5

✅ 加载 sentinel2_l2a: shape=(128, 128, 12, 12), dtype=uint16
✅ RGB波段索引: R=B04(2), G=B03(1), B=B02(0)
✅ 提取RGB: shape=(128, 128, 12, 3)
✅ 缩放因子: 4.00 (256 -> 1024)

🔄 执行超分辨率: (128, 128, 12) -> (512, 512, 12)
✅ 超分辨率完成: shape=(512, 512, 12, 3), dtype=uint16
   数据范围: [0, 65535]

💾 创建新文件: /mnt/ht2-nas2/00-model/00-jiangzf/coderepo/3996/planet_rgbnir_output/sample_0.h5
📋 复制原始数据...
✅ 原始数据复制完成
💾 写入 planet_rgbnir...
✅ 验证成功!
📊 输出文件大小: 17.91 MB

处理: sample_1.h5

超分辨率测试: sample_1.h5
模式: WRITE
输出文件: /mnt/ht2-nas2/00-model/00-jiangzf/coderepo/3996/planet_rgbnir_output/sample_1.h5

✅ 加载 sentinel2_l2a: shape=(128, 128, 12, 12), dtype=uint16
✅ RGB波段索引: R=B04(2), G=B03(1), B=B02(0)
✅ 提取RGB: shape=(128, 128, 12, 3)
✅ 缩放因子: 4.00 (256 -> 1024)

🔄 执行超分辨率: (128, 128, 12) -> (512, 512, 12)
✅ 超分辨率完成: shape=(512, 512, 12, 3), dtype=uint16
   数据范围: [0, 65535]

💾 创建新文件: /

# Pretraining with Planet RGBNIR

In [2]:
"""Trying to prototype fitting everything into olmo core."""

import sys
import os

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 强制重新加载模块（避免缓存问题）
import importlib
import olmoearth_pretrain.data.constants
importlib.reload(olmoearth_pretrain.data.constants)

import logging

from scripts.official.script import (
    build_dataloader_config,
    build_dataset_config,
    build_train_module_config,
    build_trainer_config,
)

from olmoearth_pretrain.data.constants import Modality
from olmoearth_pretrain.internal.common import (
    build_common_components as build_common_components_default,
)
from olmoearth_pretrain.internal.experiment import (
    CommonComponents,
    SubCmd,
    main,
)
from olmoearth_pretrain.internal.utils import MODEL_SIZE_ARGS
from olmoearth_pretrain.nn.flexihelios import (
    EncoderConfig,
    PredictorConfig,
)
from olmoearth_pretrain.nn.latent_mim import LatentMIMConfig

logger = logging.getLogger(__name__)

MAX_PATCH_SIZE = 8
MIN_PATCH_SIZE = 1


def build_common_components(
    script: str, cmd: SubCmd, run_name: str, cluster: str, overrides: list[str]
) -> CommonComponents:
    """Build the common components for nano_jzf experiment with custom modalities."""
    config = build_common_components_default(script, cmd, run_name, cluster, overrides)
    # 在这里添加你想要的模态，只对 nano_jzf 生效
    config.training_modalities = [
        Modality.SENTINEL2_L2A.name,
        Modality.SENTINEL1.name,
        Modality.LANDSAT.name,
        Modality.WORLDCOVER.name,
        Modality.SRTM.name,
        Modality.OPENSTREETMAP_RASTER.name,
        Modality.WRI_CANOPY_HEIGHT_MAP.name,
        Modality.CDL.name,
        Modality.WORLDCEREAL.name,
        Modality.PLANET_RGBNIR.name,
        # 在这里添加新的模态，例如：
        # Modality.YOUR_NEW_MODALITY.name,
    ]
    return config


def build_model_config(common: CommonComponents) -> LatentMIMConfig:
    """Build the model config for an experiment."""
    model_size = MODEL_SIZE_ARGS["nano"]

    encoder_config = EncoderConfig(
        embedding_size=model_size["encoder_embedding_size"],
        num_heads=model_size["encoder_num_heads"],
        depth=model_size["encoder_depth"],
        mlp_ratio=model_size["mlp_ratio"],
        supported_modality_names=common.training_modalities,
        max_patch_size=MAX_PATCH_SIZE,
        drop_path=0.1,
        max_sequence_length=12,
        use_linear_patch_embed=False,
    )
    decoder_config = PredictorConfig(
        encoder_embedding_size=model_size["encoder_embedding_size"],
        decoder_embedding_size=model_size["decoder_embedding_size"],
        depth=model_size["decoder_depth"],
        mlp_ratio=model_size["mlp_ratio"],
        num_heads=model_size["decoder_num_heads"],
        supported_modality_names=common.training_modalities,
        max_sequence_length=12,
    )
    model_config = LatentMIMConfig(
        encoder_config=encoder_config,
        decoder_config=decoder_config,
    )
    return model_config


# if __name__ == "__main__":
#     main(
#         common_components_builder=build_common_components,
#         model_config_builder=build_model_config,
#         train_module_config_builder=build_train_module_config,
#         dataset_config_builder=build_dataset_config,
#         dataloader_config_builder=build_dataloader_config,
#         trainer_config_builder=build_trainer_config,
#     )


if __name__ == "__main__":
    import sys
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    
    # 调试模式：硬编码参数
    debug_mode = True  # 调试时设为 True，正常训练时改为 False
    
    if debug_mode:
        # 直接设置 sys.argv
        sys.argv = [
            "scripts/official/nano_jzf.py",
            "train_single",
            "debug_run_new", 
            "local",
            "--dataset.h5py_dir=/mnt/ht2-nas2/00-model/00-jiangzf/coderepo/H5_DIR/h5py_data_w_missing_timesteps_zstd_3_128_x_4/cdl_landsat_openstreetmap_raster_sentinel1_sentinel2_l2a_srtm_worldcereal_worldcover_wri_canopy_height_map_planet_rgbnir/3996",
            "--data_loader.global_batch_size=640",
            "--trainer.max_duration.value=1",
            "--data_loader.num_workers=0",
            "--trainer.callbacks.wandb.enabled=False",
            "--trainer.load_strategy=never",
        ]
    
    main(
        common_components_builder=build_common_components,
        model_config_builder=build_model_config,
        train_module_config_builder=build_train_module_config,
        dataset_config_builder=build_dataset_config,
        dataloader_config_builder=build_dataloader_config,
        trainer_config_builder=build_trainer_config,
    )


KeyboardInterrupt: 